In [ ]:
import numpy as np
import joblib
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

# ──────────────────────────────────────────────
# 1. LOAD DATA
# ──────────────────────────────────────────────
land_ocean_tag = "land"

data_base_path_jan = "/work/bb1097/b383521/GW_Data/Data_set_ML/"
data_base_path_aug = "/work/bb1097/b383521/GW_Data_Summer/Data_set_ML/"

Data_Base_jan = joblib.load(data_base_path_jan + "corsed_action_" + land_ocean_tag + "_2.pkl")
Data_Base_aug = joblib.load(data_base_path_aug + "corsed_action_" + land_ocean_tag + "_2.pkl")

X_train = np.concatenate(
    [Data_Base_jan["X_train"], Data_Base_aug["X_train"]],
    axis=1,
)

X_test_raw = np.concatenate(
    [Data_Base_jan["X_test"], Data_Base_aug["X_test"]],
    axis=1,
)

y_test = np.concatenate(
    [Data_Base_jan["y_test"], Data_Base_aug["y_test"]],
    axis=1,
)
y_test = np.nan_to_num(y_test, nan=0.0)

ntri_land = Data_Base_jan["ntri"]
n_features = X_train.shape[-1]
n_targets = y_test.shape[2] * y_test.shape[3] * y_test.shape[4]  # 6*6*4 = 144

# ──────────────────────────────────────────────
# 2. BUILD ABLATED (NON-ORO) AND ORIGINAL COPIES — FIXED
# ──────────────────────────────────────────────
orographic_mean = np.median(X_train[:, :, 102:], axis=(0, 1))

X_test_oro    = X_test_raw.copy()   # untouched, real orography features
X_test_nonoro = X_test_raw.copy()   # ablated version
X_test_nonoro[:, :, 102:] = orographic_mean

# ──────────────────────────────────────────────
# 3. RESHAPE
# ──────────────────────────────────────────────
X_train       = X_train.reshape(-1, n_features)
X_test_oro    = X_test_oro.reshape(-1, n_features)
X_test_nonoro = X_test_nonoro.reshape(-1, n_features)
y_test        = y_test.reshape(-1, n_targets)

# ──────────────────────────────────────────────
# 4. SCALE (fit scaler on training data only, apply to both test versions)
# ──────────────────────────────────────────────
x_scaler = StandardScaler()
X_train       = x_scaler.fit_transform(X_train)
X_test_oro    = x_scaler.transform(X_test_oro)
X_test_nonoro = x_scaler.transform(X_test_nonoro)
del X_train, X_test_raw
# ──────────────────────────────────────────────
# 5. LOAD MODEL AND PREDICT
# ──────────────────────────────────────────────
save_dir   = "/work/bb1097/b383521/ML_models/coarsed_grained/"
ML_dir     = save_dir + "models/"
fname_model = f"cnn_coarsed_{land_ocean_tag}_2.keras"

model = tf.keras.models.load_model(
    ML_dir + fname_model,
    custom_objects=False, compile=False
)

y_test_oro_pred    = model.predict(X_test_oro)
y_test_nonoro_pred = model.predict(X_test_nonoro)

In [ ]:
from Flux_energy import compute_intrinsic_feq_cgz
cgz, intrinsic_freq, k_lunch, l_lunch, m_lunch = compute_intrinsic_feq_cgz()
y_test_nonoro_pred = y_test_nonoro_pred.reshape(len(ntri_land), 44, 144)
y_test_oro_pred    = y_test_oro_pred.reshape(len(ntri_land), 44, 144)
y_test             = y_test.reshape(len(ntri_land), 44, 144)
Energy_oro     = np.sum(np.expm1(y_test_oro_pred)*intrinsic_freq[ntri_land, ...], axis=-1)
Energ_oro_flat = np.sum(np.expm1(y_test_nonoro_pred)*intrinsic_freq[ntri_land, ...], axis=-1)
Energ_true = np.sum(y_test*intrinsic_freq[ntri_land, ...], axis=-1)

In [ ]:
Energy_oro_jan = np.mean(Energy_oro[:, :22], axis=1)
Energy_oro_flat_jan = np.mean(Energ_oro_flat[:, :22], axis=1)
Energy_oro_aug = np.mean(Energy_oro[:, 22:], axis=1)
Energy_oro_flat_aug = np.mean(Energ_oro_flat[:, 22:], axis=1)
Energy_true_jan     = np.mean(Energ_true[:, :22], axis=1)
Energy_true_aug     = np.mean(Energ_true[:, 22:], axis=1)

In [ ]:
# assumes you have E_true, E_oro, E_nonoro per-sample arrays for the FULL test set
# (both Jan and Aug), plus a way to split them by month

def summarize_ablation_by_regime(E_true, E_oro, E_nonoro, label):
    p50, p90, p99 = np.percentile(E_true, [50, 90, 99])
    regimes = {
        "P0-P50":   E_true <= p50,
        "P50-P90":  (E_true > p50)  & (E_true <= p90),
        "P90-P99":  (E_true > p90)  & (E_true <= p99),
        "P99-P100": E_true > p99,
    }
    print(f"\n--- {label} ---")
    for name, mask in regimes.items():
        oro_mean    = E_oro[mask].mean()
        nonoro_mean = E_nonoro[mask].mean()
        abs_diff    = oro_mean - nonoro_mean
        rel_diff    = abs_diff / oro_mean if oro_mean != 0 else np.nan
        print(f"{name:10s} n={mask.sum():6d}  "
              f"E_oro={oro_mean:10.1f}  E_nonoro={nonoro_mean:10.1f}  "
              f"reduction={rel_diff:.1%}")



summarize_ablation_by_regime(Energy_true_jan, Energy_oro_jan, Energy_oro_flat_jan, "January")
summarize_ablation_by_regime(Energy_true_aug, Energy_oro_aug, Energy_oro_flat_aug, "August")